# Stage 1 of 4 — Download Official House PTR PDFs From the House XML Filing Index

## What this stage actually does

This notebook builds the **source PDF archive** for the project.

For each selected year, it downloads the official House financial-disclosure XML index, reads every filing entry, keeps only records whose `FilingType` is `P` (Periodic Transaction Report), and downloads the corresponding official House PTR PDF into Google Drive.

## Why you need this stage

Every later stage depends on having the original House PDFs locally. This is the ingestion step that creates:

```text
MyDrive/
└── Congressional Trading Data/House_PTRs/
    └── 01 Official House PTR PDFs/
        ├── 2021/
        ├── 2022/
        ├── 2023/
        ├── 2024/
        ├── 2025/
        └── 2026/
```

Each PDF is stored as `<DocID>.pdf`.


## Project filesystem

The first time this notebook runs, it starts the project under:

```text
MyDrive/
└── Congressional Trading Data/House_PTRs/
    └── 01 Official House PTR PDFs/
        ├── 2021/
        ├── 2022/
        ├── 2023/
        ├── 2024/
        ├── 2025/
        └── 2026/
```

Later stages add their own XML, verification, output, checkpoint, and workflow folders beneath the same root.

## Important behavior

- **Safe to rerun:** if a PDF already exists, it is skipped.
- **Source-faithful:** PDFs are saved exactly as downloaded.
- **No transaction parsing yet:** this stage only gets the source documents.
- The XML index used here is temporary for discovery. Stage 2 separately archives the official XML itself and verifies completeness.

The code below is intentionally heavily commented so the notebook remains understandable months or years later.

## 1. Mount Google Drive

### Code walkthrough — connect Google Drive

This cell mounts your Google Drive at `/content/drive`.

Why: every downloaded PDF needs to survive after the temporary Colab runtime shuts down.

In [ ]:
# ============================================================================
# STAGE 1 — GOOGLE DRIVE
# ============================================================================
# Make the persistent MyDrive filesystem available inside this temporary Colab runtime.

from google.colab import drive
drive.mount("/content/drive")


## 2. Choose the years you want

Change this list whenever you want.

For example:

```python
YEARS = [2025, 2026]
```

### Code walkthrough — choose archive years

`YEARS` controls which House annual XML indexes and PDF folders are processed.

The current production range is 2021 through 2026 inclusive.

In [ ]:
# ============================================================================
# STAGE 1 — YEAR RANGE
# ============================================================================
# This is the only value you normally change when expanding or narrowing the archive period.

YEARS = list(range(2021, 2027))


## 3. Import the Python tools we need

### Code walkthrough — import downloader tools

- `Path` handles folders and filenames.
- `ElementTree` parses the House XML index.
- `requests` downloads XML and PDFs over HTTPS.
- `time` lets the downloader pause briefly between PDF requests.

In [ ]:
# ============================================================================
# STAGE 1 — IMPORTS
# ============================================================================
# Load only the standard/network tools needed to discover and download official House PTR source files.

from pathlib import Path
import xml.etree.ElementTree as ET
import requests
import time


## 4. Download the PTRs

For each year, this cell:

1. Downloads the House financial-disclosure XML index.
2. Looks through every filing.
3. Keeps only filings where `FilingType` is `P` (only want PTRs).
4. Gets the `DocID`.
5. Builds the PTR PDF URL.
6. Checks whether that PDF already exists in Google Drive.
7. Downloads it only if it is missing.

### Code walkthrough — discover PTRs and download PDFs

This is the main Stage 1 loop.

For each year it:

1. Downloads `<YEAR>FD.xml` from the House Clerk.
2. Parses every `<Member>` filing record.
3. Keeps only `FilingType == "P"`.
4. Reads the filing `DocID`.
5. Builds the official PTR PDF URL.
6. Skips the file if it is already in Drive.
7. Downloads missing PDFs.
8. Prints counts for listed, downloaded, and already-present files.

The existing code already contains line-by-line comments; those comments are retained.

In [ ]:
# ============================================================================
# STAGE 1 — MAIN DOWNLOADER
# ============================================================================
# Discover official PTR DocIDs from the annual House XML index and download missing source PDFs. Existing PDFs are deliberately left untouched.

# BASE_FOLDER is the main folder where all downloaded PTR PDFs will be saved.
# Path() makes it easier to work with folders and filenames in Python.
PROJECT_ROOT = Path("/content/drive/MyDrive/Congressional Trading Data/House_PTRs")
BASE_FOLDER = PROJECT_ROOT / "01 Official House PTR PDFs"

# Create the descriptive source-PDF archive folder if it does not already exist.
# parents=True means Python can also create any missing parent folders.
# exist_ok=True means don't throw an error if the folder already exists.
BASE_FOLDER.mkdir(parents=True, exist_ok=True)


# Some websites behave differently depending on what kind of browser/program
# is making the request.
#
# This tells the House website that our request looks like it came from
# a normal web browser instead of an unidentified Python script.
headers = {
    "User-Agent": "Mozilla/5.0"
}


# Loop through every year in our YEARS list.
#
# Example:
# YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
#
# First year = 2021
# then 2022
# then 2023
# etc.
for year in YEARS:

    # Print a visual divider so we can easily see which year is being processed.
    #
    # The f before the string lets us insert the value of year using {year}.
    # \n means start with a new line.
    print(f"\n===== {year} =====")


    # Build the URL for that year's House financial disclosure XML file.
    #
    # If year = 2026, this becomes:
    #
    # https://disclosures-clerk.house.gov/
    # public_disc/financial-pdfs/2026FD.xml
    xml_url = (
        f"https://disclosures-clerk.house.gov/"
        f"public_disc/financial-pdfs/{year}FD.xml"
    )


    # Just tell us what the program is doing.
    print("Downloading filing index...")


    # requests.get() downloads information from a URL.
    #
    # xml_url = the address we want
    # headers=headers = send the browser-style User-Agent above
    #
    # The result gets stored in the variable response.
    response = requests.get(xml_url, headers=headers)


    # Check whether the download succeeded.
    #
    # If the website returns something like:
    # 404 = file not found
    # 500 = server error
    #
    # this line stops the program and shows the error.
    response.raise_for_status()


    # response.content contains the actual XML file we just downloaded.
    #
    # ET = xml.etree.ElementTree
    #
    # ET.fromstring() reads the XML and turns it into a structure
    # Python can search through.
    root = ET.fromstring(response.content)


    # Create a folder specifically for this year.
    #
    # If:
    # BASE_FOLDER = Congressional Trading Data/House_PTRs/01 Official House PTR PDFs
    # year = 2026
    #
    # then year_folder becomes:
    #
    # Congressional Trading Data/House_PTRs/01 Official House PTR PDFs/2026
    year_folder = BASE_FOLDER / str(year)


    # Create the year's folder if it does not already exist.
    year_folder.mkdir(exist_ok=True)


    # These are counters so we can see what happened when the program finishes.
    #
    # Start all three at zero.
    total_ptrs = 0
    downloaded = 0
    skipped = 0


    # Look through every XML element named "Member".
    #
    # Each Member entry represents one financial disclosure filing.
    for member in root.iter("Member"):


        # Find the FilingType field inside this Member entry.
        #
        # Example:
        # <FilingType>P</FilingType>
        #
        # filing_type would then equal:
        # "P"
        filing_type = member.findtext("FilingType")


        # Find the DocID field.
        #
        # Example:
        # <DocID>20034567</DocID>
        #
        # doc_id would then equal:
        # "20034567"
        doc_id = member.findtext("DocID")


        # We only care about PTRs.
        #
        # P = Periodic Transaction Report
        #
        # != means "does not equal"
        #
        # So:
        # if filing_type is NOT "P"
        # skip the rest of this loop and move to the next filing.
        if filing_type != "P":
            continue


        # Make sure the filing actually has a DocID.
        #
        # "if not doc_id" means:
        # if doc_id is missing or empty.
        #
        # Without a DocID we cannot build the PDF URL.
        if not doc_id:
            continue


        # If we got this far, we know this filing is a PTR with a DocID.
        #
        # Add 1 to our total PTR count.
        total_ptrs += 1


        # Decide where this PDF should be saved.
        #
        # Example:
        #
        # year_folder = Congressional Trading Data/House_PTRs/01 Official House PTR PDFs/2026
        # doc_id = 20034567
        #
        # pdf_path becomes:
        #
        # Congressional Trading Data/House_PTRs/01 Official House PTR PDFs/2026/20034567.pdf
        pdf_path = year_folder / f"{doc_id}.pdf"


        # Check whether we already have this PDF.
        #
        # .exists() returns:
        # True  = file exists
        # False = file does not exist
        if pdf_path.exists():

            # Count it as a skipped file.
            skipped += 1

            # Move immediately to the next filing.
            #
            # This is what prevents us from downloading duplicates.
            continue


        # Build the URL for the actual PTR PDF.
        #
        # Example:
        #
        # year = 2026
        # doc_id = 20034567
        #
        # becomes:
        #
        # https://disclosures-clerk.house.gov/
        # public_disc/ptr-pdfs/2026/20034567.pdf
        pdf_url = (
            f"https://disclosures-clerk.house.gov/"
            f"public_disc/ptr-pdfs/{year}/{doc_id}.pdf"
        )


        # try means:
        #
        # "Attempt this code, but if something goes wrong,
        # don't crash the whole program."
        try:


            # Download the actual PDF.
            #
            # timeout=30 means:
            # if the website does not respond within 30 seconds,
            # give up on this particular request.
            pdf_response = requests.get(
                pdf_url,
                headers=headers,
                timeout=30
            )


            # Make sure the PDF download succeeded.
            #
            # If we get a 404, 500, etc., this creates an error
            # that gets handled by the except section below.
            pdf_response.raise_for_status()


            # pdf_response.content contains the actual bytes of the PDF.
            #
            # write_bytes() saves those bytes as a file in Google Drive.
            pdf_path.write_bytes(pdf_response.content)


            # We successfully downloaded one PDF,
            # so increase the downloaded counter by 1.
            downloaded += 1


            # Show us which DocID was downloaded.
            print("Downloaded:", doc_id)


            # Pause for half a second before requesting another PDF.
            #
            # This avoids hammering the House website with requests
            # as fast as Python possibly can.
            time.sleep(0.5)


        # If anything inside the try block fails,
        # Python jumps down here instead of killing the whole program.
        except Exception as error:


            # Print which DocID failed and the error message.
            #
            # Then the loop continues on to the next filing.
            print("ERROR:", doc_id, error)


    # Print a blank line after finishing the year.
    print()


    # Show how many PTR records existed in the XML.
    print("PTRs listed:", total_ptrs)


    # Show how many new PDFs we actually downloaded.
    print("Downloaded:", downloaded)


    # Show how many PDFs were already in Drive and therefore skipped.
    print("Already had:", skipped)


    # Show the folder where this year's PDFs were saved.
    print("Saved in:", year_folder)

## Where the PDFs go

Your Google Drive will look like this:

```text
My Drive
└── Congressional Trading Data
    └── House_PTRs
        └── 01 Official House PTR PDFs
            ├── 2024
            │   ├── 20012345.pdf
            │   └── ...
            ├── 2025
            │   └── ...
            └── 2026
                └── ...
```

To grab several years, just change:

```python
YEARS = [2024, 2025, 2026]
```

Then rerun the download cell.

The notebook will **not redownload files that already exist**.